# Lab 06: Function Calling and Tools (Gemini 3.1)

Function calling turns Gemini into an active agent. The `google-genai` SDK offers two ways to handle tools: **Automatic** (where the SDK executes the code for you) and **Manual** (where you control every step of the handshake).

## Objectives
1. Define Python tools with docstrings.
2. Use **Automatic Function Calling (AFC)** for fast development.
3. Use **Manual Function Calling** to inspect intermediate `FunctionCall` objects.
4. Implement a custom **execution loop**.

In [ ]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized.")

## 1. Defining Tools

Gemini uses the function's name, type hints, and **docstring** to create a JSON schema automatically.

In [ ]:
def get_stock_price(ticker: str) -> float:
    """Returns the current stock price for a given ticker symbol."""
    prices = {"GOOGL": 150.25, "AAPL": 180.50, "TSLA": 240.00}
    print(f"[Tool Execution] Fetching price for {ticker}...")
    return prices.get(ticker.upper(), 100.0)

def get_weather(city: str) -> str:
    """Returns the current weather for a city."""
    print(f"[Tool Execution] Fetching weather for {city}...")
    return f"The weather in {city} is sunny, 22°C."

tools = [get_stock_price, get_weather]

## 2. The "Magic" Way: Automatic Function Calling (AFC)

By default, when you pass Python functions to `tools`, the SDK intercepts the model's call, executes your function locally, sends the result back to Gemini, and gives you only the final text response.

In [ ]:
prompt = "What is the price of GOOGL stock?"

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(tools=tools),
    contents=prompt
)

print(f"Gemini Final Response: {response.text}")
print(
    "\nNote: You only saw the text, but the '[Tool Execution]' print above proves "
    "the function was called automatically."
)

## 3. The Manual Way: Disabling AFC

To see what happens "under the hood", we can disable AFC. This is useful if you want to validate tool calls before execution or if your tools run on a different server.

In [ ]:
prompt = "What is the weather in Paris?"

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        tools=tools,
        # Disable automatic execution
        automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
    ),
    contents=prompt
)

print("Response Parts (Manual Mode):")
for part in response.candidates[0].content.parts:
    if part.function_call:
        print(f"- Function Call: {part.function_call.name}")
        print(f"- Arguments: {part.function_call.args}")
    if part.text:
        print(f"- Text Part: {part.text}")

## 4. Manual Execution Loop

In a manual workflow, you are responsible for the handshake. This is the pattern used by complex agent frameworks like LangChain or AutoGen.

In [ ]:
def run_manual_agent(user_query):
    print(f"User: {user_query}\n")

    # 1. Ask Gemini (Manual Mode)
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        config=types.GenerateContentConfig(
            tools=tools,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)
        ),
        contents=user_query
    )

    # 2. Extract and Execute Calls
    tool_results = []
    for part in response.candidates[0].content.parts:
        if part.function_call:
            fn_name = part.function_call.name
            args = part.function_call.args

            # Execute locally
            if fn_name == "get_stock_price":
                result = get_stock_price(**args)
            elif fn_name == "get_weather":
                result = get_weather(**args)

            # Create response part
            tool_results.append(types.Part.from_function_response(
                name=fn_name, response={"result": result}
            ))

    # 3. Send results back to get final text
    if tool_results:
        final_response = client.models.generate_content(
            model="gemini-3.1-flash-lite-preview",
            contents=[
                types.Content(
                    role="user", parts=[types.Part.from_text(text=user_query)]
                ),
                response.candidates[0].content, # The Call part
                types.Content(role="user", parts=tool_results) # The Response part
            ]
        )
        print(f"\nGemini Final Answer: {final_response.text}")

run_manual_agent("What is the weather in London and the price of AAPL?")

## 5. Declaring Tools via JSON Schemas

If your tool is not a Python function (e.g., a remote API) or if you want absolute control over the definition, you can pass a **JSON Schema** directly. The SDK won't be able to auto-execute this (no AFC), so you must handle it manually.

In [ ]:
# Define a tool using a dictionary (OpenAPI-like schema)
json_tool = {
    "name": "find_nearest_restaurant",
    "description": "Finds the nearest restaurant based on cuisine and budget.",
    "parameters": {
        "type": "object",
        "properties": {
            "cuisine": {
                "type": "string",
                "description": "The type of food (e.g., Italian, Sushi)"
            },
            "budget": {
                "type": "number",
                "description": "Maximum price in USD"
            }
        },
        "required": ["cuisine"]
    }
}

tools = types.Tool(function_declarations=[json_tool])

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        tools=[tools], # Pass the raw schema
    ),
    contents="I'm hungry for some Sushi and I have $50."
)

print("Response Parts (Manual Mode):")
for part in response.candidates[0].content.parts:
    if part.function_call:
        print(f"- Function Call: {part.function_call.name}")
        print(f"- Arguments: {part.function_call.args}")
    if part.text:
        print(f"- Text Part: {part.text}")

## Summary

In this lab, you learned the three ways to handle Function Calling:
1. **AFC (Automatic)**: SDK handles the schema and the execution. Best for Python tools.
2. **Manual (Python)**: SDK handles the schema, you handle the execution. Best for auditing/validation.
3. **JSON Schema**: You handle the schema and the execution. Best for non-Python tools or remote APIs.